## Pipeline model - Spaceship Titanic Kaggle Competition

### Importing Libraries

In [24]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

### Loading and Combining Data

In [25]:
train_path = 'train.csv'
test_path = 'test.csv'

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

y = train_data['Transported']
train_data = train_data.drop(columns=['Transported'])

combined = pd.concat([train_data, test_data], keys=['train', 'test'])
combined

PassengerId HomePlanet CryoSleep     Cabin    Destination   Age  \
train 0        0001_01     Europa     False     B/0/P    TRAPPIST-1e  39.0   
      1        0002_01      Earth     False     F/0/S    TRAPPIST-1e  24.0   
      2        0003_01     Europa     False     A/0/S    TRAPPIST-1e  58.0   
      3        0003_02     Europa     False     A/0/S    TRAPPIST-1e  33.0   
      4        0004_01      Earth     False     F/1/S    TRAPPIST-1e  16.0   
...                ...        ...       ...       ...            ...   ...   
test  4272     9266_02      Earth      True  G/1496/S    TRAPPIST-1e  34.0   
      4273     9269_01      Earth     False       NaN    TRAPPIST-1e  42.0   
      4274     9271_01       Mars      True   D/296/P    55 Cancri e   NaN   
      4275     9273_01     Europa     False   D/297/P            NaN   NaN   
      4276     9277_01      Earth      True  G/1498/S  PSO J318.5-22  43.0   

              VIP  RoomService  FoodCourt  ShoppingMall     Spa  VRDeck  \
train 0     False          0.0        0.0           0.0     0.0     0.0   
      1     False        109.0        9.0          25.0   549.0    44.0   
      2      True         43.0     3576.0           0.0  6715.0    49.0   
      3     False          0.0     1283.0         371.0  3329.0   193.0   
      4     False        303.0       70.0         151.0   565.0     2.0   
...           ...          ...        ...           ...     ...     ...   
test  4272  False          0.0        0.0           0.0     0.0     0.0   
      4273  False          0.0      847.0          17.0    10.0   144.0   
      4274  False          0.0        0.0           0.0     0.0     0.0   
      4275  False          0.0     2680.0           0.0     0.0   523.0   
      4276  False          0.0        0.0           0.0     0.0     0.0   

                         Name  
train 0       Maham Ofracculy  
      1          Juanna Vines  
      2         Altark Susent  
      3          Solam Susent  
      4     Willy Santantines  
...                       ...  
test  4272        Jeron Peter  
      4273      Matty Scheron  
      4274        Jayrin Pore  
      4275     Kitakan Conale  
      4276   Lilace Leonzaley  

[12970 rows x 13 columns]

#### This is the conbination overview of the training data and the testing data, conbining training and test sets help to avoid unknown features in the test data

### Calculating Total Spending

In [26]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

combined['TotalSpending'] = combined[spend_cols].sum(axis=1)

### Splitting the Cabin Column in to deck and side

In [27]:
cabin_split = combined['Cabin'].str.split('/', expand=True)

combined['Deck'] = cabin_split[0]
combined['Side'] = cabin_split[2]
combined['Side'].value_counts(dropna=False)

Side
S      6381
P      6290
NaN     299
Name: count, dtype: int64

#### There are 299 NaN in my side data which I need to fix on

### Selecting Features and filling Unknown for Missing Values

In [28]:
features = [
    'CryoSleep', 'Age', 'VIP', 'RoomService', 'FoodCourt',
    'ShoppingMall', 'Spa', 'VRDeck', 'Deck', 'Side',
    'TotalSpending'
]

X_train_full = combined.loc['train', features]
X_test_full = combined.loc['test', features]

X_train_full = (
    X_train_full.fillna({'Deck': 'Unknown', 'Side': 'Unknown'})
    .fillna(0)
)

X_test_full = (
    X_test_full.fillna({'Deck': 'Unknown', 'Side': 'Unknown'})
    .fillna(0)
)

### Splitting the Training Data for Validation

In [29]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y, test_size=0.2, random_state=0
)

### Defining Categorical and Numerical Columns

In [30]:
categorical_cols = ['Deck', 'Side']

numerical_cols = [
    'CryoSleep', 'VIP', 'TotalSpending', 'Age',
    'Spa', 'VRDeck', 'RoomService', 'FoodCourt', 'ShoppingMall'
]

### Creating a Pipeline model

In [31]:
numerical_transformer = SimpleImputer(strategy='constant', fill_value=0)

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

### Training the XGBoost Model Pipeline

In [32]:
xgb_model = XGBClassifier(
    n_estimators=1300,
    learning_rate=0.05,
    n_jobs=4,
    random_state=0
)

my_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

my_pipeline.fit(X_train, y_train)

preds = my_pipeline.predict(X_valid)
score = accuracy_score(y_valid, preds)

print('Validation Accuracy:', score)

Validation Accuracy: 0.7947096032202415


#### The current pridicted Accuracy is around 79.4%

### Training on Full Data and getting Submission CSV file

In [33]:
my_pipeline.fit(X_train_full, y)

test_preds = my_pipeline.predict(X_test_full)
test_preds_bool = test_preds.astype(bool)

output = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Transported': test_preds_bool
})

submission_file_name = f'submission_{pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M-%S")}.csv'
output.to_csv(submission_file_name, index=False)